# [SQL 실습 #06] MySQL 데이터 수정하기
## UPDATE와 SET 기초

> **학생용 실습 노트북 - TODO 완성형**

지난 시간에는 `SELECT`와 `WHERE`를 이용하여 원하는 데이터를 찾아보았습니다.  
이번 시간에는 이미 저장된 데이터를 수정하는 `UPDATE`를 배웁니다.

## 오늘의 목표

1. `UPDATE` 기본 문법을 이해한다.
2. `SET`으로 수정할 컬럼과 값을 지정할 수 있다.
3. `WHERE`로 수정 대상을 정확하게 선택할 수 있다.
4. 수정 전 `SELECT`로 대상을 먼저 확인하는 습관을 익힌다.
5. 한 개의 컬럼과 여러 개의 컬럼을 수정할 수 있다.
6. 이름보다 `id` 같은 고유값을 조건으로 사용하는 이유를 이해한다.
7. 조건에 맞는 여러 행을 한 번에 수정할 수 있다.
8. `WHERE`가 없는 `UPDATE`가 왜 위험한지 설명할 수 있다.
9. `NULL` 값을 실제 값으로 수정할 수 있다.
10. 기존 값을 이용해 `grade = grade + 1`처럼 수정할 수 있다.

> **핵심**
>
> `UPDATE`는 이미 저장된 데이터를 고치는 명령어입니다.  
> `SET`은 고칠 내용, `WHERE`는 고칠 대상을 지정합니다.

# 1. MySQL 실습 환경 준비

PDF에서는 이전 시간에 만든 `school` 데이터베이스를 이어서 사용합니다.

Google Colab에서는 런타임이 새로 시작될 때 MySQL 환경이 없어질 수 있으므로,
아래 셀에서 MySQL 서버를 준비합니다.

> 아래 셀은 수정하지 말고 실행하세요.

In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start

print("✅ MySQL 서버 준비 완료")

# 2. `school.students` 테이블 준비

PDF #06의 실습 데이터는 다음 8개 행으로 시작합니다.

| id | name | grade | class_name |
|---:|---|---:|---|
| 1 | 김부장 | 3 | 컴퓨터과 |
| 2 | 혼이 | 1 | 영혼반 |
| 3 | 라즈베리 | 2 | 임베디드반 |
| 4 | 리눅스 | 3 | 서버반 |
| 5 | 파이썬 | 1 | 프로그래밍반 |
| 6 | 김코딩 | 1 | 컴퓨터과 |
| 7 | 데이터왕 | 2 | 데이터베이스반 |
| 8 | 정보없음 | NULL | NULL |

PDF 화면과 맞추기 위해 `created_at`은 `2026-08-01 20:00:00`으로 고정합니다.

아래 셀은 **수업 시작용 초기화 셀**입니다.

In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name, created_at)
VALUES
('김부장', 3, '컴퓨터과', '2026-08-01 20:00:00'),
('혼이', 1, '영혼반', '2026-08-01 20:00:00'),
('라즈베리', 2, '임베디드반', '2026-08-01 20:00:00'),
('리눅스', 3, '서버반', '2026-08-01 20:00:00'),
('파이썬', 1, '프로그래밍반', '2026-08-01 20:00:00'),
('김코딩', 1, '컴퓨터과', '2026-08-01 20:00:00'),
('데이터왕', 2, '데이터베이스반', '2026-08-01 20:00:00');

INSERT INTO students (name, created_at)
VALUES ('정보없음', '2026-08-01 20:00:00');
SQL

echo "✅ PDF #06 기준 실습 데이터 준비 완료"

# 3. 안전한 SQL 실행 / 자동점검 도우미

이번 단원은 `UPDATE`를 실제로 실행하므로 안전장치를 넣었습니다.

### 학생 TODO에서 적용되는 안전 규칙

- `UPDATE`에는 반드시 `WHERE`가 있어야 합니다.
- `WHERE 1=1`, `WHERE TRUE`처럼 사실상 전체 행을 선택하는 조건은 차단합니다.
- `DROP`, `TRUNCATE`, `DELETE`, `ALTER`는 학생 TODO에서 실행할 수 없습니다.
- `INSERT`는 PDF의 `미정학생` 실습에서만 별도 함수로 실행합니다.
- 수정 후에는 테이블 전체 상태를 확인하여 엉뚱한 행까지 바뀌지 않았는지 점검합니다.

> **중요:** 이 셀은 수정하지 마세요.

In [ ]:
import subprocess
import textwrap
import base64
import re

TODO_STATUS = {}
LAST_SQL = ""
LAST_MODE = ""

_EXPECTED_SELECTS_B64 = {1: 'VVNFIHNjaG9vbDsgU0hPVyBUQUJMRVM7', 2: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50czs=', 3: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lID0gJ+q5gOy9lOuUqSc7', 5: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lID0gJ+2YvOydtCc7', 7: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lID0gJ+ygleuztOyXhuydjCc7', 9: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBpZCA9IDQ7', 11: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lID0gJ+y7tO2TqO2EsOqzvCc7', 13: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBpZCA9IDY7', 16: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lID0gJ+uvuOygle2VmeyDnSc7', 18: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBpZCA9IDE7', 20: 'VVNFIHNjaG9vbDsgU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lID0gJ+y7tO2TqO2EsOqzte2VmeqzvCc7'}
_EXPECTED_STAGES = {'base': [['1', '김부장', '3', '컴퓨터과'], ['2', '혼이', '1', '영혼반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '1', '컴퓨터과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', 'NULL', 'NULL']], 'kimcoding_grade2': [['1', '김부장', '3', '컴퓨터과'], ['2', '혼이', '1', '영혼반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '2', '컴퓨터과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', 'NULL', 'NULL']], 'honi_dbclass': [['1', '김부장', '3', '컴퓨터과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '2', '컴퓨터과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', 'NULL', 'NULL']], 'unknown_filled': [['1', '김부장', '3', '컴퓨터과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '2', '컴퓨터과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터과']], 'linux_class': [['1', '김부장', '3', '컴퓨터과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '2', '컴퓨터과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터과']], 'multirow_computer_engineering': [['1', '김부장', '3', '컴퓨터공학과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '2', '컴퓨터공학과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터공학과']], 'id6_grade3': [['1', '김부장', '3', '컴퓨터공학과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '3', '컴퓨터공학과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터공학과']], 'pending_inserted': [['1', '김부장', '3', '컴퓨터공학과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '3', '컴퓨터공학과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터공학과'], ['9', '미정학생', 'NULL', 'NULL']], 'pending_filled': [['1', '김부장', '3', '컴퓨터공학과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '3', '컴퓨터공학과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터공학과'], ['9', '미정학생', '1', '미정반']], 'id1_plus1': [['1', '김부장', '4', '컴퓨터공학과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '3', '컴퓨터공학과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터공학과'], ['9', '미정학생', '1', '미정반']], 'class_string_restored': [['1', '김부장', '4', '컴퓨터과'], ['2', '혼이', '1', '데이터베이스반'], ['3', '라즈베리', '2', '임베디드반'], ['4', '리눅스', '3', '리눅스서버반'], ['5', '파이썬', '1', '프로그래밍반'], ['6', '김코딩', '3', '컴퓨터과'], ['7', '데이터왕', '2', '데이터베이스반'], ['8', '정보없음', '2', '컴퓨터과'], ['9', '미정학생', '1', '미정반']]}

def _mysql(sql, no_header=False):
    cmd = ["sudo", "mysql", "--batch", "--raw"]
    if no_header:
        cmd += ["-N"]
    result = subprocess.run(
        cmd,
        input=textwrap.dedent(sql).strip(),
        text=True,
        capture_output=True
    )
    return result.returncode == 0, result.stdout.strip(), result.stderr.strip()

def _real_sql(sql):
    lines = [
        line for line in textwrap.dedent(sql).splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ]
    return "\n".join(lines).strip()

def _contains_todo(sql):
    return "TODO" in _real_sql(sql)

def run_select(sql):
    global LAST_SQL, LAST_MODE
    sql = textwrap.dedent(sql).strip()
    LAST_SQL, LAST_MODE = sql, "select"

    if not _real_sql(sql):
        print("⚠️ SQL이 비어 있습니다.")
        return False
    if _contains_todo(sql):
        print("⚠️ TODO가 남아 있습니다. 먼저 완성하세요.")
        return False

    forbidden = [r"\bUPDATE\b", r"\bDELETE\b", r"\bDROP\b",
                 r"\bTRUNCATE\b", r"\bALTER\b", r"\bINSERT\b"]
    if any(re.search(p, sql, re.I) for p in forbidden):
        print("❌ 이 셀은 조회 전용입니다.")
        return False

    ok, out, err = _mysql(sql)
    if out:
        print(out)
    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False
    print("✅ 조회 완료")
    return True

def _validate_update(sql):
    real = _real_sql(sql)
    if not real:
        return False, "SQL이 비어 있습니다."
    if "TODO" in real:
        return False, "TODO가 남아 있습니다."
    if len(re.findall(r"\bUPDATE\b", real, re.I)) != 1:
        return False, "학생 TODO에서는 UPDATE 한 문장만 실행하세요."
    if not re.search(r"\bSET\b", real, re.I):
        return False, "SET이 없습니다."
    if not re.search(r"\bWHERE\b", real, re.I):
        return False, "WHERE가 없습니다. 전체 행 수정 위험 때문에 실행을 차단했습니다."
    if re.search(r"\bWHERE\s+(1\s*=\s*1|TRUE)\b", real, re.I):
        return False, "모든 행과 일치하는 WHERE 조건은 차단합니다."
    forbidden = [r"\bDELETE\b", r"\bDROP\b", r"\bTRUNCATE\b",
                 r"\bALTER\b", r"\bINSERT\b"]
    if any(re.search(p, real, re.I) for p in forbidden):
        return False, "UPDATE 이외의 변경 명령이 함께 들어 있습니다."
    return True, ""

def run_update(sql):
    global LAST_SQL, LAST_MODE
    sql = textwrap.dedent(sql).strip()
    LAST_SQL, LAST_MODE = sql, "update"

    ok_safe, msg = _validate_update(sql)
    if not ok_safe:
        print("❌ 안전장치:", msg)
        return False

    ok, out, err = _mysql(sql)
    if out:
        print(out)
    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False

    print("✅ UPDATE 실행 완료")
    return True

def run_insert(sql):
    global LAST_SQL, LAST_MODE
    sql = textwrap.dedent(sql).strip()
    LAST_SQL, LAST_MODE = sql, "insert"

    real = _real_sql(sql)
    if not real or "TODO" in real:
        print("⚠️ TODO가 남아 있거나 SQL이 비어 있습니다.")
        return False
    if len(re.findall(r"\bINSERT\b", real, re.I)) != 1:
        print("❌ 이 실습에서는 INSERT 한 문장만 실행하세요.")
        return False
    forbidden = [r"\bUPDATE\b", r"\bDELETE\b", r"\bDROP\b",
                 r"\bTRUNCATE\b", r"\bALTER\b"]
    if any(re.search(p, real, re.I) for p in forbidden):
        print("❌ INSERT 이외의 변경 명령이 들어 있습니다.")
        return False

    ok, out, err = _mysql(sql)
    if out:
        print(out)
    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False
    print("✅ INSERT 실행 완료")
    return True

def _decode_expected_select(todo_id):
    return base64.b64decode(
        _EXPECTED_SELECTS_B64[todo_id]
    ).decode("utf-8")

def _normalize_table_output(text):
    lines = [line.rstrip() for line in text.splitlines() if line.strip()]
    if not lines:
        return []
    header = lines[0]
    rows = sorted(lines[1:])
    return [header] + rows

def check_select(todo_id, sql, current_todo_id=None):
    print(f"🔎 TODO {todo_id} 자동점검")

    if current_todo_id != todo_id:
        TODO_STATUS[todo_id] = False
        print("❌ 바로 위 TODO 셀을 먼저 실행하세요.")
        return False
    if _contains_todo(sql):
        TODO_STATUS[todo_id] = False
        print("❌ TODO가 남아 있습니다.")
        return False

    ok_s, out_s, err_s = _mysql(sql)
    if not ok_s:
        TODO_STATUS[todo_id] = False
        print("❌ 학생 SQL 실행 오류")
        print(err_s)
        return False

    expected = _decode_expected_select(todo_id)
    ok_e, out_e, err_e = _mysql(expected)
    if not ok_e:
        TODO_STATUS[todo_id] = False
        print("⚠️ 자동점검 기준 SQL 오류")
        return False

    passed = _normalize_table_output(out_s) == _normalize_table_output(out_e)
    TODO_STATUS[todo_id] = passed

    if passed:
        print("✅ 통과: 조회 결과가 정확합니다.")
    else:
        print("❌ 재확인: 조회 결과가 요구사항과 다릅니다.")
    return passed

def _current_snapshot():
    query = '''
    SELECT id, name, grade, class_name
    FROM school.students
    ORDER BY id;
    '''
    ok, out, err = _mysql(query, no_header=True)
    if not ok:
        raise RuntimeError(err)

    rows = []
    for line in out.splitlines():
        if line.strip():
            rows.append(line.split("\t"))
    return rows

def check_stage(todo_id, stage_name, current_todo_id=None):
    print(f"🔎 TODO {todo_id} 자동점검")

    if current_todo_id != todo_id:
        TODO_STATUS[todo_id] = False
        print("❌ 바로 위 TODO 셀을 먼저 실행하세요.")
        return False

    actual = _current_snapshot()
    expected = _EXPECTED_STAGES[stage_name]

    passed = actual == expected
    TODO_STATUS[todo_id] = passed

    if passed:
        print("✅ 통과: 수정 결과와 다른 행의 상태까지 정확합니다.")
    else:
        print("❌ 재확인: 예상하지 않은 행까지 수정되었거나 값이 다릅니다.")
        print("💡 실습 데이터가 꼬였다면 맨 아래 '실습 초기화' 셀을 실행한 뒤 처음부터 진행하세요.")
    return passed

def explain_update_safety(sql):
    print("검사할 SQL:")
    print(textwrap.dedent(sql).strip())
    print()

    ok, msg = _validate_update(sql)
    if ok:
        print("✅ 안전장치 기준: WHERE가 있는 UPDATE입니다.")
    else:
        print("🚫 실행 차단:", msg)

def show_progress(total=21):
    done = sum(TODO_STATUS.get(i) is True for i in range(1, total + 1))
    failed = sum(TODO_STATUS.get(i) is False for i in range(1, total + 1))
    unchecked = total - done - failed

    print("=" * 54)
    print("📊 SQL 실습 #06 TODO 진행 현황")
    print("=" * 54)

    for start in range(1, total + 1, 7):
        end = min(start + 6, total)
        parts = []
        for i in range(start, end + 1):
            state = TODO_STATUS.get(i)
            mark = "✅" if state is True else "❌" if state is False else "⬜"
            parts.append(f"{i}:{mark}")
        print("   ".join(parts))

    print("-" * 54)
    print(f"통과       : {done}/{total}")
    print(f"재확인 필요: {failed}/{total}")
    print(f"미점검     : {unchecked}/{total}")
    print(f"진행률     : {done / total * 100:.1f}%")

# 4. 사용할 데이터베이스 선택하기


먼저 `school` 데이터베이스를 선택하고 `students` 테이블이 있는지 확인합니다.


## TODO 1
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE TODO;
SHOW TODO;
"""
_todo_id = 1
run_select(sql)

In [ ]:
# TODO 1 자동점검 - 수정하지 마세요.
check_select(1, sql, globals().get("_todo_id"))

# 5. 실습 데이터 전체 확인하기


UPDATE를 시작하기 전에 전체 데이터를 먼저 확인합니다.

PDF에서는 **수정 전 데이터를 먼저 보는 습관**을 강조합니다.


## TODO 2
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT TODO FROM students;
"""
_todo_id = 2
run_select(sql)

In [ ]:
# TODO 2 자동점검 - 수정하지 마세요.
check_select(2, sql, globals().get("_todo_id"))

# 6. UPDATE 기본 문법

기본 형태는 다음과 같습니다.

```sql
UPDATE 테이블명
SET 컬럼명 = 새값
WHERE 조건;
```

쉽게 말하면:

```text
어느 테이블을 수정할지 정하고,
어떤 컬럼을 어떤 값으로 바꿀지 정하고,
어떤 행을 수정할지 조건으로 고른다.
```

이번 예제에서는 `김코딩` 학생의 학년을 **1학년 → 2학년**으로 수정합니다.

# 7. 수정 전에는 SELECT로 먼저 확인하기


먼저 `김코딩` 학생이 정확히 한 명 조회되는지 확인하세요.

이 과정이 UPDATE 안전 습관의 첫 단계입니다.


## TODO 3
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name = 'TODO';
"""
_todo_id = 3
run_select(sql)

In [ ]:
# TODO 3 자동점검 - 수정하지 마세요.
check_select(3, sql, globals().get("_todo_id"))

# 8. 김코딩 학생의 학년 수정하기


`김코딩`의 `grade`를 `2`로 수정하세요.

PDF의 핵심 예제입니다.


## TODO 4
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET TODO = TODO
WHERE name = 'TODO';
"""
_todo_id = 4
run_update(sql)

In [ ]:
# TODO 4 자동점검 - 수정하지 마세요.
check_stage(4, "kimcoding_grade2", globals().get("_todo_id"))

In [ ]:
print("===== 수정 결과 확인 =====")
ok, out, err = _mysql("USE school; SELECT * FROM students WHERE name = '김코딩';")
print(out if ok else err)

# 9. 한 개의 컬럼 수정하기 - 혼이 확인


이번에는 `혼이`의 학과명을 수정합니다.

현재 `혼이`의 `class_name`이 `영혼반`인지 먼저 확인하세요.


## TODO 5
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name = 'TODO';
"""
_todo_id = 5
run_select(sql)

In [ ]:
# TODO 5 자동점검 - 수정하지 마세요.
check_select(5, sql, globals().get("_todo_id"))

# 10. 한 개의 컬럼 수정하기 - 혼이 학과명 변경


`혼이`의 `class_name`을 `데이터베이스반`으로 수정하세요.


## TODO 6
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET TODO = 'TODO'
WHERE name = 'TODO';
"""
_todo_id = 6
run_update(sql)

In [ ]:
# TODO 6 자동점검 - 수정하지 마세요.
check_stage(6, "honi_dbclass", globals().get("_todo_id"))

# 11. 여러 컬럼을 한 번에 수정하기 - 정보없음 확인


`정보없음` 학생은 `grade`와 `class_name`이 모두 `NULL`입니다.

먼저 현재 상태를 확인하세요.


## TODO 7
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name = 'TODO';
"""
_todo_id = 7
run_select(sql)

In [ ]:
# TODO 7 자동점검 - 수정하지 마세요.
check_select(7, sql, globals().get("_todo_id"))

# 12. 여러 컬럼을 한 번에 수정하기


`정보없음` 학생의 두 컬럼을 한 번에 수정합니다.

- `grade = 2`
- `class_name = '컴퓨터과'`

여러 컬럼은 `SET` 뒤에서 **쉼표(,)** 로 구분합니다.


## TODO 8
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET grade = TODO, class_name = 'TODO'
WHERE name = 'TODO';
"""
_todo_id = 8
run_update(sql)

In [ ]:
# TODO 8 자동점검 - 수정하지 마세요.
check_stage(8, "unknown_filled", globals().get("_todo_id"))

# 13. WHERE 조건에 id 사용하기 - 수정 대상 확인


이름은 중복될 수 있지만 `id`는 각 행을 구분하는 고유값입니다.

PDF에서는 **id가 4번인 학생**을 수정하는 예제를 사용합니다.
먼저 해당 학생을 확인하세요.


## TODO 9
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE id = TODO;
"""
_todo_id = 9
run_select(sql)

In [ ]:
# TODO 9 자동점검 - 수정하지 마세요.
check_select(9, sql, globals().get("_todo_id"))

# 14. WHERE 조건에 id 사용하기 - id 4 수정


`id = 4`인 학생의 `class_name`을 `리눅스서버반`으로 수정하세요.


## TODO 10
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET class_name = 'TODO'
WHERE id = TODO;
"""
_todo_id = 10
run_update(sql)

In [ ]:
# TODO 10 자동점검 - 수정하지 마세요.
check_stage(10, "linux_class", globals().get("_todo_id"))

# 15. 조건에 맞는 여러 행 수정하기 - 대상 확인


UPDATE는 한 행만 수정하는 명령이 아닙니다.

`class_name = '컴퓨터과'`인 학생이 현재 몇 명인지 먼저 조회하세요.
PDF에서는 이 조건에 맞는 행이 **3개**입니다.


## TODO 11
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name = 'TODO';
"""
_todo_id = 11
run_select(sql)

In [ ]:
# TODO 11 자동점검 - 수정하지 마세요.
check_select(11, sql, globals().get("_todo_id"))

# 16. 조건에 맞는 여러 행 한 번에 수정하기


`class_name`이 `컴퓨터과`인 모든 행을 `컴퓨터공학과`로 수정하세요.

조건에 맞는 행이 여러 개라면 여러 행이 한 번에 수정될 수 있습니다.


## TODO 12
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET class_name = 'TODO'
WHERE class_name = 'TODO';
"""
_todo_id = 12
run_update(sql)

In [ ]:
# TODO 12 자동점검 - 수정하지 마세요.
check_stage(12, "multirow_computer_engineering", globals().get("_todo_id"))

# 17. WHERE 없이 UPDATE를 실행하면?

다음 SQL은 **실행하면 안 되는 위험한 예**입니다.

```sql
UPDATE students
SET grade = 1;
```

`WHERE`가 없으므로 `students` 테이블의 모든 행이 수정됩니다.

PDF에서도 이 예제는 설명만 하고 실제 실행하지 않는 것을 권장합니다.

아래 셀은 SQL을 **실행하지 않고 안전성만 검사**합니다.

In [ ]:
unsafe_sql = '''
UPDATE students
SET grade = 1;
'''

explain_update_safety(unsafe_sql)

## 안전한 UPDATE 습관

UPDATE를 사용할 때 다음 순서를 습관처럼 지킵니다.

1. `SELECT`로 수정할 대상 먼저 확인
2. `WHERE` 조건을 정확히 작성
3. `UPDATE` 실행
4. 다시 `SELECT`로 결과 확인

> **SELECT → UPDATE → SELECT**

# 18. 안전한 UPDATE 습관 - id 6 확인


PDF에서는 `id = 6`인 학생을 먼저 조회한 다음 같은 조건으로 수정합니다.

먼저 대상 학생을 확인하세요.


## TODO 13
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE id = TODO;
"""
_todo_id = 13
run_select(sql)

In [ ]:
# TODO 13 자동점검 - 수정하지 마세요.
check_select(13, sql, globals().get("_todo_id"))

# 19. 안전한 UPDATE 습관 - id 6 학년 수정


`id = 6`인 학생의 학년을 `3`으로 수정하세요.


## TODO 14
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET grade = TODO
WHERE id = TODO;
"""
_todo_id = 14
run_update(sql)

In [ ]:
# TODO 14 자동점검 - 수정하지 마세요.
check_stage(14, "id6_grade3", globals().get("_todo_id"))

In [ ]:
print("===== UPDATE 후 다시 SELECT =====")
ok, out, err = _mysql("USE school; SELECT * FROM students WHERE id = 6;")
print(out if ok else err)

# 20. NULL 값을 수정하기

앞 단계에서 `정보없음`의 NULL 값을 이미 채웠기 때문에,
PDF에서는 새로운 학생 `미정학생`을 한 명 추가한 뒤 NULL 수정 실습을 합니다.

`name`만 입력하면 `grade`와 `class_name`은 `NULL`로 남습니다.

# 21. NULL 실습용 미정학생 추가


`students` 테이블에 이름이 `미정학생`인 행을 추가하세요.

이번 INSERT는 UPDATE 실습을 위한 준비 단계입니다.


## TODO 15
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
INSERT INTO students (name)
VALUES ('TODO');
"""
_todo_id = 15
run_insert(sql)

In [ ]:
# TODO 15 자동점검 - 수정하지 마세요.
check_stage(15, "pending_inserted", globals().get("_todo_id"))

# 22. NULL 값 확인하기


추가한 `미정학생`의 `grade`와 `class_name`이 `NULL`인지 확인하세요.


## TODO 16
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name = 'TODO';
"""
_todo_id = 16
run_select(sql)

In [ ]:
# TODO 16 자동점검 - 수정하지 마세요.
check_select(16, sql, globals().get("_todo_id"))

# 23. NULL 값을 실제 값으로 수정하기


`미정학생`의 빈 정보를 다음과 같이 채우세요.

- `grade = 1`
- `class_name = '미정반'`


## TODO 17
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET grade = TODO,
    class_name = 'TODO'
WHERE name = 'TODO';
"""
_todo_id = 17
run_update(sql)

In [ ]:
# TODO 17 자동점검 - 수정하지 마세요.
check_stage(17, "pending_filled", globals().get("_todo_id"))

# 24. 기존 값을 활용해서 수정하기 - id 1 확인


UPDATE의 오른쪽에는 기존 컬럼 값을 사용할 수도 있습니다.

PDF에서는 `id = 1` 학생의 현재 학년을 확인한 뒤
`grade = grade + 1`을 사용합니다.


## TODO 18
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE id = TODO;
"""
_todo_id = 18
run_select(sql)

In [ ]:
# TODO 18 자동점검 - 수정하지 마세요.
check_select(18, sql, globals().get("_todo_id"))

# 25. 기존 값에 1 더하기


`id = 1` 학생의 학년을 현재 값에서 **1 증가**시키세요.

직접 `4`를 넣지 말고 PDF처럼 기존 값을 활용하세요.


## TODO 19
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET grade = grade TODO TODO
WHERE id = TODO;
"""
_todo_id = 19
run_update(sql)

In [ ]:
# TODO 19 자동점검 - 수정하지 마세요.
check_stage(19, "id1_plus1", globals().get("_todo_id"))

# 26. 문자열 값 수정하기 - 대상 확인


PDF에서는 `컴퓨터공학과`라는 문자열 값을 다시 `컴퓨터과`로 바꿉니다.

먼저 `class_name = '컴퓨터공학과'`인 행들을 확인하세요.


## TODO 20
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name = 'TODO';
"""
_todo_id = 20
run_select(sql)

In [ ]:
# TODO 20 자동점검 - 수정하지 마세요.
check_select(20, sql, globals().get("_todo_id"))

# 27. 문자열 값 수정하기


`class_name = '컴퓨터공학과'`인 행들의 값을 `컴퓨터과`로 수정하세요.

이 예제는 문자열의 **일부를 함수로 치환하는 것이 아니라**,
WHERE로 특정 문자열 값을 찾은 뒤 새 문자열 값으로 수정하는 실습입니다.


## TODO 21
아래 `TODO`를 알맞게 완성한 뒤 실행하세요.

> UPDATE는 실제 데이터를 바꿉니다. 조건을 다시 확인하세요.

In [ ]:
sql = """
USE school;
UPDATE students
SET class_name = 'TODO'
WHERE class_name = 'TODO';
"""
_todo_id = 21
run_update(sql)

In [ ]:
# TODO 21 자동점검 - 수정하지 마세요.
check_stage(21, "class_string_restored", globals().get("_todo_id"))

# 28. 자주 발생하는 실수

## 1) WHERE를 빼먹는 경우

위험한 예:

```sql
UPDATE students
SET grade = 1;
```

→ 모든 학생의 학년이 수정될 수 있습니다.

안전한 예:

```sql
UPDATE students
SET grade = 1
WHERE id = 6;
```

---

## 2) 문자열에 작은따옴표를 쓰지 않는 경우

잘못된 예:

```sql
UPDATE students
SET class_name = 컴퓨터과
WHERE name = 김부장;
```

올바른 예:

```sql
UPDATE students
SET class_name = '컴퓨터과'
WHERE name = '김부장';
```

---

## 3) `=`와 `==`를 헷갈리는 경우

SQL 비교에서는 `=`를 사용합니다.

잘못된 예:

```sql
WHERE grade == 3;
```

올바른 예:

```sql
WHERE grade = 3;
```

UPDATE에서도 마찬가지입니다.

```sql
UPDATE students
SET grade = 3
WHERE id = 1;
```

---

## 4) NULL을 `=`로 비교하는 경우

잘못된 예:

```sql
WHERE grade = NULL;
```

올바른 예:

```sql
WHERE grade IS NULL;
```

NULL 값을 수정할 때도 먼저 `IS NULL`로 대상을 확인하는 것이 좋습니다.

```sql
UPDATE students
SET grade = 1
WHERE grade IS NULL;
```

단, 이 조건은 NULL인 행이 여러 개라면 여러 행을 수정할 수 있으므로
**먼저 SELECT로 대상 개수를 확인해야 합니다.**

# 29. 오늘 실습 코드 전체 정리

아래는 PDF #06의 실습 흐름을 순서대로 정리한 것입니다.

```sql
USE school;

TRUNCATE TABLE students;

INSERT INTO students (name, grade, class_name)
VALUES
('김부장', 3, '컴퓨터과'),
('혼이', 1, '영혼반'),
('라즈베리', 2, '임베디드반'),
('리눅스', 3, '서버반'),
('파이썬', 1, '프로그래밍반'),
('김코딩', 1, '컴퓨터과'),
('데이터왕', 2, '데이터베이스반');

INSERT INTO students (name)
VALUES ('정보없음');

SELECT * FROM students;

SELECT * FROM students
WHERE name = '김코딩';

UPDATE students
SET grade = 2
WHERE name = '김코딩';

SELECT * FROM students
WHERE name = '김코딩';

UPDATE students
SET class_name = '데이터베이스반'
WHERE name = '혼이';

UPDATE students
SET grade = 2, class_name = '컴퓨터과'
WHERE name = '정보없음';

UPDATE students
SET class_name = '리눅스서버반'
WHERE id = 4;

UPDATE students
SET class_name = '컴퓨터공학과'
WHERE class_name = '컴퓨터과';

SELECT * FROM students
WHERE id = 6;

UPDATE students
SET grade = 3
WHERE id = 6;

SELECT * FROM students
WHERE id = 6;

INSERT INTO students (name)
VALUES ('미정학생');

UPDATE students
SET grade = 1, class_name = '미정반'
WHERE name = '미정학생';

UPDATE students
SET grade = grade + 1
WHERE id = 1;

UPDATE students
SET class_name = '컴퓨터과'
WHERE class_name = '컴퓨터공학과';
```

> **주의:** `UPDATE students SET grade = 1;`처럼 WHERE가 없는 UPDATE는
> 테이블 전체를 수정할 수 있으므로 실습에서도 실행하지 않습니다.

# 30. 전체 TODO 진행 현황

아래 셀에서 TODO 1~21의 자동점검 결과와 현재 테이블을 확인하세요.

In [ ]:
show_progress(21)

print("\n===== 현재 students 상태 =====")
ok, out, err = _mysql(
    "SELECT id, name, grade, class_name FROM school.students ORDER BY id;"
)
print(out if ok else err)

# 31. 오늘 배운 내용 정리

```text
UPDATE → 기존 데이터를 수정하는 명령어
SET    → 어떤 컬럼을 어떤 값으로 바꿀지 지정
WHERE  → 수정할 대상을 고르는 조건
```

### 꼭 기억할 점

- WHERE 없이 UPDATE를 실행하면 전체 행이 수정될 수 있다.
- 수정 전에는 SELECT로 대상을 먼저 확인한다.
- 수정 후에도 SELECT로 결과를 다시 확인한다.
- 여러 컬럼은 SET에서 쉼표로 구분한다.
- 이름보다 id 같은 고유값이 더 정확한 수정 조건이 될 수 있다.
- 조건에 맞는 행이 여러 개면 여러 행이 한 번에 수정된다.
- NULL은 `= NULL`이 아니라 `IS NULL`로 찾는다.
- 기존 값을 이용해 `grade = grade + 1`과 같은 수정도 가능하다.

> **안전한 습관:** `SELECT → UPDATE → SELECT`

# 32. 자기 점검

- [ ] UPDATE의 역할을 설명할 수 있다.
- [ ] SET과 WHERE의 역할을 구분할 수 있다.
- [ ] 수정 전에 SELECT로 대상을 확인할 수 있다.
- [ ] 한 개 컬럼을 수정할 수 있다.
- [ ] 여러 컬럼을 한 번에 수정할 수 있다.
- [ ] id를 조건으로 정확한 행을 수정할 수 있다.
- [ ] 조건에 맞는 여러 행이 동시에 수정될 수 있음을 안다.
- [ ] WHERE 없는 UPDATE가 위험한 이유를 설명할 수 있다.
- [ ] NULL 값을 실제 값으로 수정할 수 있다.
- [ ] `grade = grade + 1`의 의미를 설명할 수 있다.
- [ ] 문자열 값도 UPDATE로 수정할 수 있다.

# 33. [선택] 실습을 처음 상태로 되돌리기

실습 중 SQL을 잘못 실행했거나 순서가 꼬였다면
`RESET = True`로 바꾸고 실행하세요.

PDF #06 시작 상태의 8개 행으로 복원되며 TODO 진행 상태도 초기화됩니다.

In [ ]:
RESET = False

if RESET:
    reset_sql = '''
    USE school;
    TRUNCATE TABLE students;

    INSERT INTO students (name, grade, class_name, created_at)
    VALUES
    ('김부장', 3, '컴퓨터과', '2026-08-01 20:00:00'),
    ('혼이', 1, '영혼반', '2026-08-01 20:00:00'),
    ('라즈베리', 2, '임베디드반', '2026-08-01 20:00:00'),
    ('리눅스', 3, '서버반', '2026-08-01 20:00:00'),
    ('파이썬', 1, '프로그래밍반', '2026-08-01 20:00:00'),
    ('김코딩', 1, '컴퓨터과', '2026-08-01 20:00:00'),
    ('데이터왕', 2, '데이터베이스반', '2026-08-01 20:00:00');

    INSERT INTO students (name, created_at)
    VALUES ('정보없음', '2026-08-01 20:00:00');
    '''

    ok, out, err = _mysql(reset_sql)
    if ok:
        TODO_STATUS.clear()
        print("✅ PDF #06 시작 상태로 복원했습니다.")
        print("✅ TODO 진행 상태도 초기화했습니다.")
    else:
        print("❌ 초기화 오류")
        print(err)
else:
    print("ℹ️ RESET = False 입니다. 데이터는 변경하지 않았습니다.")

# 다음 시간 예고

## [SQL 실습 #07] MySQL 데이터 삭제하기
### DELETE와 TRUNCATE 차이

다음 시간에는 데이터를 삭제하는 방법을 학습합니다.

- 특정 행만 삭제하는 `DELETE`
- 테이블 전체 데이터를 비우는 `TRUNCATE`
- 실수로 전체 데이터를 삭제하지 않기 위한 주의사항